# Deploy a basic agent

This notebook deploys a simple kagent agent without tools that uses Anthropic Claude through a `ModelConfig`. See the [README](./README.md) for other model providers and the same steps in a terminal.

kagent must be enabled on your prokube cluster, and you need an Anthropic API key.

## Setup

Set up this repository's kagent helpers from [kagent_helpers.py](../shared/kagent_helpers.py). Adding their directory to `sys.path` lets the notebook import them without installing anything.

In [ ]:
import sys

sys.path.insert(0, "../shared")  # kagent helpers (shared by all kagent examples)

import kagent_helpers

In [ ]:
import os
import subprocess
from getpass import getpass
from pathlib import Path

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()


def kubectl_apply(manifest: str) -> None:
    subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

## Create the credentials

The agents call Claude with the key in the `anthropic-api-key` Secret. The next cell creates the Secret unless it already exists, for example under **K8s Secrets** in the prokube user menu (see [Kubernetes Secrets](https://prokube.ai/docs/1.9/platform/kubernetes.html#kubernetes-secrets)). It prompts for the key unless `ANTHROPIC_API_KEY` is set.

In [ ]:
secret_exists = subprocess.run(
    ["kubectl", "get", "secret", "anthropic-api-key"], capture_output=True
).returncode == 0
if not secret_exists:
    anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")
    kubectl_apply(
        Path("anthropic-secret.yaml")
        .read_text()
        .replace("<your-anthropic-api-key>", anthropic_api_key)
    )

## Deploy

Create the ModelConfig and the agent:

In [ ]:
!kubectl apply -f anthropic-model.yaml
!kubectl wait --for=condition=Accepted modelconfigs.kagent.dev/anthropic-haiku --timeout=2m
!kubectl apply -f assistant.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/simple-assistant --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open **Agents** in the prokube UI (see [Agents](https://prokube.ai/docs/1.9/agentops/agents.html)), select `simple-assistant`, and chat there.

Expect a short description of what the assistant can help with.

In [ ]:
internal_url = kagent_helpers.workspace_a2a_url(namespace, "simple-assistant")
print(internal_url)

In [ ]:
prompt = "What can you help me with?"
print(kagent_helpers.send_message(internal_url, prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/simple-assistant`, with an API key. See [Public vs. Internal Traffic](https://prokube.ai/docs/1.9/platform/agent_gateway.html#public-vs-internal-traffic) for how this route differs from the internal one.

Enter your prokube domain, the host name you open prokube at, for example `my-cluster.prokube.cloud`. The cell uses `PROKUBE_DOMAIN` instead if it is set:

In [ ]:
domain = os.environ.get("PROKUBE_DOMAIN") or input("prokube domain (e.g. my-cluster.prokube.cloud): ")

Create a key with access to `simple-assistant` on the [API Keys page](/pkui/ai-gateway/keys) (`https://<your-prokube-domain>/pkui/ai-gateway/keys`), then paste it when prompted below. The cell uses `A2A_API_KEY` instead if it is set:

In [ ]:
api_key = os.environ.get("A2A_API_KEY") or getpass("Agent API key: ")

Send the same prompt through the agent's external URL:

In [ ]:
api_key_url = f"https://{domain}/svc/a2a/{namespace}/simple-assistant"
print(api_key_url)

In [ ]:
print(kagent_helpers.send_message(api_key_url, prompt, api_key=api_key))

## Clean up

The commands are commented out because the next examples reuse the Secret and ModelConfig. Uncomment them and run the cell to delete the agent, ModelConfig, and Secret.

In [ ]:
# !kubectl delete -f assistant.yaml
# !kubectl delete -f anthropic-model.yaml
# !kubectl delete -f anthropic-secret.yaml